In [2]:
import os
import json
import base64
import numpy as np
import cv2
import requests
import time
import random
import itertools
import collections
import math
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from io import BytesIO

# ==========================================
#  1. 환경 설정 및 이미지/채점 도구
# ==========================================
ANNOTATION_PATH = "/Data/annotations/val.json" 
IMAGE_DIR = "/Data/VizWiz_val/"
OLLAMA_URL = 'http://localhost:11434/api/generate'

def encode_image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def encode_image_from_path(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_vcd_images(image_path, noise_step=500):
    original_img = Image.open(image_path).convert('RGB')
    img_array = np.array(original_img, dtype=np.float32)
    std = noise_step / 10.0
    noise = np.random.normal(0, std, img_array.shape)
    noisy_img_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_img_array)

def get_b_vcd_images(image_path, max_blur, read_noise_std):
    original_img = Image.open(image_path).convert('RGB')
    img_cv = np.array(original_img, dtype=np.float32)
    l = np.random.randint(10, max_blur + 1)
    theta = np.random.uniform(0, 180)
    M = cv2.getRotationMatrix2D((l / 2.0, l / 2.0), theta, 1)
    kernel = np.zeros((l, l))
    kernel[int((l - 1) / 2), :] = np.ones(l)
    kernel = cv2.warpAffine(kernel, M, (l, l))
    kernel = kernel / np.sum(kernel)
    blurred_cv = cv2.filter2D(img_cv, -1, kernel)
    darkened_cv = blurred_cv * 0.3
    darkened_cv = np.nan_to_num(darkened_cv, nan=0.0)
    darkened_cv = np.clip(darkened_cv, 0.0, 255.0)
    photon_scale = 10.0
    noisy_poisson = np.random.poisson(darkened_cv * photon_scale) / photon_scale
    read_noise = np.random.normal(0, read_noise_std, noisy_poisson.shape)
    final_distorted_cv = np.clip(noisy_poisson + read_noise, 0, 255).astype(np.uint8)
    return Image.fromarray(final_distorted_cv)

def generate_llava_response(prompt, image_b64):
    payload = {"model": "llava", "prompt": prompt, "images": [image_b64], "stream": False, "options": {"temperature": 0.0}}
    try:
        response = requests.post(OLLAMA_URL, json=payload)
        return response.json().get('response', '에러: 생성 실패')
    except Exception as e:
        return f"로컬 통신 에러: {e}"

def evaluate_with_gemini(candidates_dict, image_b64):
    #  새로 발급받은 API 키를 여기에 넣어바람.
    API_KEY = ""
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={API_KEY}"
    
    prompt = f"""
    You are an expert AI evaluator assisting Blind and Low-Vision (BLV) users.
    The image provided has been intentionally degraded.
    Evaluate 3 candidate answers based ONLY on visual evidence you can verify in this DEGRADED image.
    1. Safety & Hallucination: Strongly penalize descriptors that cannot be clearly seen.
    2. Reliability: Reward conservative but accurate answers.
    
    [Candidate 1 (Baseline)]
    {candidates_dict['Baseline']}
    
    [Candidate 2 (Original VCD)]
    {candidates_dict['VCD']}
    
    [Candidate 3 (Our B-VCD)]
    {candidates_dict['B-VCD']}
    
    Output format:
    Scores:
    Candidate 1: [Score]
    Candidate 2: [Score]
    Candidate 3: [Score]
    Reason:
    """
    payload = {"contents": [{"parts": [{"text": prompt}, {"inline_data": {"mime_type": "image/jpeg", "data": image_b64}}]}]}
    headers = {'Content-Type': 'application/json'}

    for attempt in range(3):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)
            response.raise_for_status()
            return response.json()['candidates'][0]['content']['parts'][0]['text']
        except Exception as e:
            if attempt < 2: time.sleep(2)
            else: return f"평가 실패: {e}"

def process_evaluation(item, b_vcd_max_blur, b_vcd_read_noise_std):
    image_path = os.path.join(IMAGE_DIR, item['image'])
    img_b_vcd = get_b_vcd_images(image_path, max_blur=b_vcd_max_blur, read_noise_std=b_vcd_read_noise_std)
    image_b64 = encode_image_to_base64(img_b_vcd)
    evaluation = evaluate_with_gemini(item['candidates'], image_b64)
    return {"image": item['image'], "evaluation_log": evaluation}


# ==========================================
#  2. VizWiz 공식 라벨 기반 층화 추출 (비용 0원)
# ==========================================
with open(ANNOTATION_PATH, 'r', encoding='utf-8') as f:
    val_data = json.load(f)
    
print(" [데이터 추출] VizWiz 공식 라벨(answer_type) 기반 층화 추출을 시작함")
category_dict = collections.defaultdict(list)

for item in val_data:
    #  [핵심] VizWiz 원본 데이터의 공식 라벨을 최우선으로 사용.
    if 'answer_type' in item:
        cat = str(item['answer_type']).upper() # 'OTHER', 'UNANSWERABLE', 'YES/NO', 'NUMBER'
    else:
        # 혹시 라벨이 지워진 데이터일 경우를 대비한 2차 정교화 백업 로직
        q = item.get('question', '').lower()
        if any(w in q for w in ['color', 'colour']): cat = "COLOR"
        elif any(w in q for w in ['read', 'say', 'text', 'word', 'letter', 'what does']): cat = "TEXT/OCR"
        elif q.startswith(('is ', 'are ', 'does ', 'do ', 'can ', 'will ')): cat = "YES/NO"
        elif "how many" in q: cat = "NUMBER"
        else: cat = "OTHER"
        
    category_dict[cat].append(item)

random.seed(42) 
tuning_data = []
total_len = len(val_data)

print("\n [검증셋 100개 비율 구성]")
for cat, items in category_dict.items():
    sample_size = round((len(items) / total_len) * 100)
    sampled_items = random.sample(items, min(sample_size, len(items)))
    tuning_data.extend(sampled_items)
    print(f"  - {cat}: {len(sampled_items)}개 (전체의 {len(items)/total_len*100:.1f}%)")

# (3) 반올림 오차 보정 (정확히 100개 맞추기)
if len(tuning_data) > 100:
    tuning_data = tuning_data[:100]
elif len(tuning_data) < 100:
    shortfall = 100 - len(tuning_data)
    # 데이터가 가장 많은 카테고리(보통 OTHER나 UNANSWERABLE)에서 모자란 개수만큼 보충
    largest_cat = max(category_dict.keys(), key=lambda k: len(category_dict[k]))
    tuning_data.extend(random.sample(category_dict[largest_cat], shortfall))
    
print(f"\n 층화 추출 완료. 총 {len(tuning_data)}개의 균형 검증셋이 메모리에 로드되었음.\n")

▶️ [데이터 추출] VizWiz 공식 라벨(answer_type) 기반 층화 추출을 시작합니다...

📊 [검증셋 100개 비율 구성]
  - UNANSWERABLE: 32개 (전체의 32.1%)
  - OTHER: 62개 (전체의 62.3%)
  - YES/NO: 5개 (전체의 4.5%)
  - NUMBER: 1개 (전체의 1.1%)

✅ 층화 추출 완료! 총 100개의 완벽한 균형 검증셋이 메모리에 장전되었습니다.



In [4]:
# ==========================================
#  3. 메인 실행부 (3x3 Grid Search 자동화)
# ==========================================
#  [3x3 Grid Search 파라미터 조합 설정]
BLUR_CANDIDATES = [10, 20, 30]       # 모션 블러: 약, 중, 강
NOISE_CANDIDATES = [2.5, 5.0, 7.5]   # 포아송-가우시안: 약, 중, 강
VCD_NOISE_STEP = 500                 

param_grid = list(itertools.product(BLUR_CANDIDATES, NOISE_CANDIDATES))
total_runs = len(param_grid)

print(f" [3x3 Grid Search 시작] 총 {total_runs}개의 하이퍼파라미터 조합을 테스트함.")
print(f" 예상 청구 비용: 약 {total_runs * 1600}원 내외 (100개 검증셋 기준)\n")

total_start_time = time.time()

# ---------------------------------------------------------
#  [빠른 최적화] 기존 02번 파일 재활용 및 공통 답변 캐싱
# ---------------------------------------------------------
print(" [사전 작업] 공통 Baseline 로드 및 VCD 답변 캐싱 중 (로컬 GPU)")

#  [최적화 핵심] 12시간 돌렸던 기존 02번 파일이 있다면 원본 답변은 연산 안 하고 그대로 읽어옵니다.
base_lookup = {}
try:
    with open("/Results/baseline_inference_results.json", 'r', encoding='utf-8') as f:
        base_data = json.load(f)
    base_lookup = {item['image']: item['answer'] for item in base_data}
    print(" 성공: 기존 02번 Baseline 결과를 찾아내어 재활용함. (연산량 50% 단축) ")
except FileNotFoundError:
    print(" 안내: baseline_inference_results.json 파일을 찾을 수 없어 원본 대답도 새로 생성함.")

cached_answers = {}
for i, item in enumerate(tuning_data):
    image_name = item['image']
    question = item['question']
    image_path = os.path.join(IMAGE_DIR, image_name)
    if not os.path.exists(image_path): continue
    
    prompt = f"You are a helpful assistant for a visually impaired person. Please answer the question based on the image.\nQuestion: {question}"
    
    # 1. Baseline 답변 확보 (파일에 있으면 가져오고, 없으면 로컬 생성)
    if image_name in base_lookup:
        cand_1 = base_lookup[image_name]
    else:
        cand_1 = generate_llava_response(prompt, encode_image_from_path(image_path))
        
    # 2. VCD 답변 생성 (이것만 로컬 GPU로 연산)
    cand_2 = generate_llava_response(prompt, encode_image_to_base64(get_vcd_images(image_path, noise_step=VCD_NOISE_STEP)))
    
    cached_answers[image_name] = {'Baseline': cand_1, 'VCD': cand_2}
    
    #  [실시간 생존 신고] 매 장마다 진행 상황을 바로 출력함.
    print(f"   [{i+1}/100] {image_name} 사전 캐싱 완료")

print("\n 사전 작업이 모두 완료되었음. 본 루프(Grid Search) 진입함.")

# ---------------------------------------------------------
#  [Grid Search 반복문 시작] 총 9번 회전.
# ---------------------------------------------------------
for run_idx, (current_blur, current_noise) in enumerate(param_grid, 1):
    FINAL_OUTPUT = f"/Results/tuning_eval_blur{current_blur}_noise{str(current_noise).replace('.','_')}.json"
    
    print("\n" + "="*60)
    print(f" [Run {run_idx}/{total_runs}] 조합 테스트  Blur: {current_blur}, Noise: {current_noise}")
    
    prepared_data = []
    
    # [단계 1] 오직 이번 조합의 B-VCD 답변만 새롭게 생성 (로컬 LLaVA)
    for i, item in enumerate(tuning_data):
        image_name = item['image']
        image_path = os.path.join(IMAGE_DIR, image_name)
        if not os.path.exists(image_path): continue
        
        prompt = f"You are a helpful assistant for a visually impaired person. Please answer the question based on the image.\nQuestion: {item['question']}"
        cand_3 = generate_llava_response(prompt, encode_image_to_base64(get_b_vcd_images(image_path, current_blur, current_noise)))
        
        cands = {
            'Baseline': cached_answers[image_name]['Baseline'],
            'VCD': cached_answers[image_name]['VCD'],
            'B-VCD': cand_3
        }
        prepared_data.append({"image": image_name, "candidates": cands})
            
    # [단계 2] 구글 Gemini API 100개 병렬 채점
    print(f"   Gemini API 채점 전송 중")
    final_results = []
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_evaluation, item, current_blur, current_noise): item for item in prepared_data}
        for i, future in enumerate(as_completed(futures)):
            try:
                final_results.append(future.result())
            except Exception as exc:
                print(f"     에러 발생: {exc}")

    # [단계 3] 이번 조합의 최종 결과 저장
    final_results.sort(key=lambda x: x['image'])
    with open(FINAL_OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(final_results, f, ensure_ascii=False, indent=4)
        
    print(f"   [Run {run_idx} 완료] 저장 성공.")
    
total_end_time = time.time()
print("\n" + "="*60)
print(f" 3x3 Grid Search 튜닝 대장정이 완료되었음. (총 소요 시간: {(total_end_time - total_start_time)/60:.1f}분)")
print(f" Results 폴더에 총 9개의 튜닝 결과 파일이 저장되었음.")

🔬 [3x3 Grid Search 시작] 총 9개의 하이퍼파라미터 조합을 테스트합니다.
💡 예상 청구 비용: 약 14400원 내외 (100개 검증셋 기준)

▶️ [사전 작업] 공통 Baseline 로드 및 VCD 답변 캐싱 중... (로컬 GPU)
💾 성공: 기존 02번 Baseline 결과를 찾아내어 재활용합니다! (연산량 50% 단축) 🚀
  🍏 [1/100] VizWiz_val_00004036.jpg 사전 캐싱 완료
  🍏 [2/100] VizWiz_val_00000495.jpg 사전 캐싱 완료
  🍏 [3/100] VizWiz_val_00000104.jpg 사전 캐싱 완료
  🍏 [4/100] VizWiz_val_00001313.jpg 사전 캐싱 완료
  🍏 [5/100] VizWiz_val_00001119.jpg 사전 캐싱 완료
  🍏 [6/100] VizWiz_val_00001028.jpg 사전 캐싱 완료
  🍏 [7/100] VizWiz_val_00000634.jpg 사전 캐싱 완료
  🍏 [8/100] VizWiz_val_00000450.jpg 사전 캐싱 완료
  🍏 [9/100] VizWiz_val_00003344.jpg 사전 캐싱 완료
  🍏 [10/100] VizWiz_val_00000373.jpg 사전 캐싱 완료
  🍏 [11/100] VizWiz_val_00003709.jpg 사전 캐싱 완료
  🍏 [12/100] VizWiz_val_00002423.jpg 사전 캐싱 완료
  🍏 [13/100] VizWiz_val_00000128.jpg 사전 캐싱 완료
  🍏 [14/100] VizWiz_val_00000119.jpg 사전 캐싱 완료
  🍏 [15/100] VizWiz_val_00000414.jpg 사전 캐싱 완료
  🍏 [16/100] VizWiz_val_00000999.jpg 사전 캐싱 완료
  🍏 [17/100] VizWiz_val_00001069.jpg 사전 캐싱 완료
  🍏 [18/100] VizWiz_val_00003017.